# Foundation Models в Computer Vision

В этом ноутбуке мы разберём **Foundation Models** — большие предобученные модели, которые стали основой современного компьютерного зрения.

**Что ты изучишь:**
1. CLIP — zero-shot классификация и поиск по тексту
2. DINOv2 — мощные визуальные признаки без разметки
3. SAM (Segment Anything) — универсальная сегментация
4. Feature extraction и similarity search
5. Практические примеры

> **Запускай в Colab** — Runtime → T4 GPU.

## Что такое Foundation Models?

Это большие модели, обученные на огромных датасетах (миллиарды изображений + текст). Они умеют:
- Понимать изображения **без дообучения**
- Работать с **текстом + изображением**
- Давать очень сильные признаки для любых задач

Примеры: **CLIP, DINOv2, SAM, LLaVA, Florence-2**

# 1. CLIP — Zero-Shot Classification

In [ ]:
!pip install --quiet open-clip-torch

In [ ]:
import torch
import open_clip
from PIL import Image
import requests
from io import BytesIO

# Загружаем CLIP
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
tokenizer = open_clip.get_tokenizer('ViT-B-32')

model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

In [ ]:
# Zero-shot классификация
image = Image.open(BytesIO(requests.get('https://ultralytics.com/images/bus.jpg').content))
image = preprocess(image).unsqueeze(0).to(device)

texts = ["a photo of a bus", "a photo of a car", "a photo of a dog"]
text = tokenizer(texts).to(device)

with torch.no_grad():
    image_features = model.encode_image(image)
    text_features = model.encode_text(text)
    
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)
    
    similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)

print("Вероятности:", similarity.cpu().numpy())

# 2. DINOv2 — Мощные визуальные признаки

In [ ]:
from transformers import AutoImageProcessor, AutoModel

processor = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
model = AutoModel.from_pretrained('facebook/dinov2-base').to(device)

In [ ]:
# Получение признаков
image = Image.open(BytesIO(requests.get('https://ultralytics.com/images/bus.jpg').content))
inputs = processor(images=image, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)
    features = outputs.last_hidden_state.mean(dim=1)  # глобальный признак

print(f"Размер признака: {features.shape}")  # [1, 768]

# 3. SAM — Segment Anything

In [ ]:
!pip install --quiet git+https://github.com/facebookresearch/segment-anything.git

In [ ]:
from segment_anything import sam_model_registry, SamPredictor

sam = sam_model_registry["vit_b"](checkpoint="sam_vit_b_01ec64.pth")
sam.to(device)
predictor = SamPredictor(sam)

# Пример использования (нужно передать точки или bounding box)
predictor.set_image(image)
masks, scores, logits = predictor.predict(
    point_coords=None,
    point_labels=None,
    box=None,
    multimask_output=True
)

# 4. Similarity Search (поиск похожих изображений)

In [ ]:
# Используем DINOv2 или CLIP для построения базы признаков
# Затем ищем ближайшие соседи с помощью cosine similarity или FAISS

# 5. Упражнения

### Упражнение 1: Zero-shot классификация на своём датасете

Возьми 5–10 классов и проверь, как хорошо CLIP справляется без дообучения.

In [ ]:
# Твой код здесь

### Упражнение 2: Построй простой поиск по изображению

Используй DINOv2 или CLIP для поиска похожих изображений в папке.

In [ ]:
# Твой код здесь

### Упражнение 3: SAM + YOLO

Сначала найди объекты с помощью YOLO, затем сегментируй их с помощью SAM.

---

**Готово!**  
Ты познакомился с самыми мощными моделями современного компьютерного зрения.

Следующий шаг — **WS9: Classical Computer Vision** (SIFT, ORB, RANSAC и т.д.).